In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sim_ranking as sr
import ml_tools as mlt

In [ ]:
nzgmdb_ffp = Path("/Users/claudy/dev/work/data/gm_datasets/nz_gmdb/v4.3/custom/mod_ground_motion_im_table_rotd50_flat.csv")
gnn_result_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/0326_1434_cv_v4p3NZGMDB_v2p7_6e8s")

In [ ]:
run_config = sr.ml.RunConfig.from_yaml(gnn_result_dir / "run_config.yaml")

obs_data = sr.data.load_obs_nzgmdb(nzgmdb_ffp)
gnn_results = pd.read_parquet(gnn_result_dir / "val_results.parquet").sort_index()

cim_results = None
if (gnn_result_dir / "cim_results").exists():
    cim_results = pd.read_parquet(gnn_result_dir / "cim_results" / "val_results.parquet").sort_index()
    assert cim_results.index.equals(gnn_results.index), "CIM results and GNN results do not match in index"

print(f"Number of scenarios {gnn_results.shape[0]}")

### Number Of Predictions/Records Per Site of Interest

In [ ]:
# Number of predictions per site
n_pred_per_site = gnn_results.site_int.value_counts().sort_values()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.bar(np.arange(len(n_pred_per_site)), n_pred_per_site.values)
ax.set_xlim(0, len(n_pred_per_site))
ax.set_ylabel("Number of predictions/records")

fig.tight_layout()

## Site-Pairs

In [ ]:
# Get the site-pairs with at least 10 predictions/records
print("Getting site pairs")
site_pairs_count = {}
site_pair_events = {}
for cur_ix, cur_row in gnn_results.iterrows():
    cur_site_int = cur_row.site_int
    cur_obs_sites = cur_row.obs_sites

    cur_site_pairs = mlt.array_utils.numpy_str_join("_", cur_site_int, cur_obs_sites)

    for cur_site_pair in cur_site_pairs:
        if cur_site_pair not in site_pairs_count:
            site_pairs_count[cur_site_pair] = 1
            site_pair_events[cur_site_pair] = [cur_row.event_id]
        else:
            site_pairs_count[cur_site_pair] += 1
            site_pair_events[cur_site_pair].append(cur_row.event_id)

site_pairs_df = pd.DataFrame.from_dict(site_pairs_count, orient="index", columns=["count"])
site_pairs_df["site_int"] = site_pairs_df.index.str.split("_").str[0]
site_pairs_df["obs_site"] = site_pairs_df.index.str.split("_").str[1]

# Drop any with less than 10 predictions/recordings
site_pairs_df = site_pairs_df.loc[site_pairs_df["count"] >= 10]


#### Number of events per site-pair

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.bar(np.arange(site_pairs_df.shape[0]), site_pairs_df["count"].sort_values().values)
ax.set_xlim(0, site_pairs_df.shape[0])
ax.set_ylabel("Number of events per site pair")

fig.tight_layout()


#### Number of site-pairs per site of interest

In [ ]:
site_int_n_pairs =  site_pairs_df["site_int"].value_counts()

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(np.arange(len(site_int_n_pairs)), site_int_n_pairs.sort_values().values)
ax.set_xlim(0, len(site_int_n_pairs))
ax.set_ylabel("Number of site pairs per site")

fig.tight_layout()

#### Site-to-site distance of site pairs

In [ ]:
dist_matrix = sr.utils.calculate_distance_matrix(obs_data.sites, obs_data.site_df)


In [ ]:
site_int_ind = dist_matrix.index.get_indexer_for(site_pairs_df["site_int"].values)
obs_site_ind = dist_matrix.columns.get_indexer_for(site_pairs_df["obs_site"].values)

site_dist = dist_matrix.values[site_int_ind, obs_site_ind]
site_pairs_df["dist"] = site_dist


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.hist(site_pairs_df["dist"].values, bins=50)
ax.set_xlim(0, site_pairs_df["dist"].max())
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Number of site pairs")

fig.tight_layout()

## Site-Correlations

In [ ]:
site_int_pred_im_df = gnn_results[run_config.pred_mean_keys].rename(columns=dict(zip(run_config.pred_mean_keys, run_config.ims)))
obs_im_df = np.log(obs_data.record_df[run_config.ims])

gnn_pred_site_pair_corrs = sr.ml.gnn_gm.compute_site_int_obs_correlations(site_pairs_df, site_pair_events, run_config.ims, site_int_pred_im_df, obs_im_df, verbose=False)
obs_site_pair_corrs = sr.ml.gnn_gm.compute_site_int_obs_correlations(site_pairs_df, site_pair_events, run_config.ims, obs_im_df, obs_im_df, verbose=False)

cim_pred_site_pair_corrs = None 
if cim_results is not None:
    cim_pred_im_df = cim_results[sr.constants.CIM_PRED_PSA_KEYS].rename(columns=dict(zip(sr.constants.CIM_PRED_PSA_KEYS, run_config.ims)))
    cim_pred_site_pair_corrs = sr.ml.gnn_gm.compute_site_int_obs_correlations(site_pairs_df, site_pair_events, run_config.ims, cim_pred_im_df, obs_im_df, verbose=False)
    

assert np.all(gnn_pred_site_pair_corrs.index == obs_site_pair_corrs.index)
assert np.all(gnn_pred_site_pair_corrs.index == site_pairs_df.index)

### IM Correlation With Respect To Site-To-Site Distance

In [ ]:
plot_ims = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]

if run_config.im_set == sr.constants.IMSet.all:
    plot_ims.append("PGA")
    plot_ims.append("CAV")

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), n_cols=2, n_rows=-1, ind_figsize=(8, 6))

for ix,  (cur_ax, cur_im) in enumerate(zip(axs, plot_ims)):

    cur_ax.scatter(site_pairs_df["dist"].values, gnn_pred_site_pair_corrs[cur_im].values, label="GNN Predicted", c="blue", s=5)
    cur_ax.scatter(site_pairs_df["dist"].values, obs_site_pair_corrs[cur_im].values, label="Observed", c="red", s=5)
    if cim_pred_site_pair_corrs is not None:
        cur_ax.scatter(site_pairs_df["dist"].values, cim_pred_site_pair_corrs[cur_im].values, label="CIM Predicted", c="green", s=5)

    cur_ax.set_xlim(0, site_pairs_df["dist"].max())
    cur_ax.set_ylim(0, 1)
    cur_ax.set_xlabel("Distance (km)")
    cur_ax.set_ylabel("Correlation")

    cur_ax.set_title(cur_im)

    if ix == 0:
        cur_ax.legend()    


fig.tight_layout()

### Correlation Residuals

In [ ]:
gnn_corr_residuals = obs_site_pair_corrs - gnn_pred_site_pair_corrs

cim_corr_residuals = None
if cim_pred_site_pair_corrs is not None:
    cim_corr_residuals = obs_site_pair_corrs - cim_pred_site_pair_corrs

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), n_cols=2, n_rows=-1, ind_figsize=(8, 6))

for ix,  (cur_ax, cur_im) in enumerate(zip(axs, plot_ims)):
    cur_ax.scatter(site_pairs_df["dist"].values, gnn_corr_residuals[cur_im].values, label="GNN Residuals", c="blue", s=5)
    if cim_corr_residuals is not None:
        cur_ax.scatter(site_pairs_df["dist"].values, cim_corr_residuals[cur_im].values, label="CIM Residuals", c="green", s=5)

    cur_ax.set_xlim(0, site_pairs_df["dist"].max())
    cur_ax.set_ylim(-1, 1)
    cur_ax.set_xlabel("Distance (km)")
    cur_ax.set_ylabel("Correlation residual")

    cur_ax.set_title(cur_im)

fig.tight_layout()

In [ ]:
gnn_corr_res_bias = gnn_corr_residuals.mean(axis=0)
gnn_corr_res_std = gnn_corr_residuals.std(axis=0)

cim_corr_res_bias, cim_corr_res_std = None, None
if cim_corr_residuals is not None:
    cim_corr_res_bias = cim_corr_residuals.mean(axis=0)
    cim_corr_res_std = cim_corr_residuals.std(axis=0)
    

fig, ax1, ax2 = sr.plot_utils.get_pSA_bias_residual_fig()

ax1.plot(sr.constants.PERIODS, gnn_corr_res_bias[sr.constants.PSA_KEYS], label="Bias", c="blue")
ax2.plot(sr.constants.PERIODS, gnn_corr_res_std[sr.constants.PSA_KEYS], label="Std", c="blue")

if cim_corr_res_bias is not None:
    ax1.plot(sr.constants.PERIODS, cim_corr_res_bias[sr.constants.PSA_KEYS], label="Bias", c="green")
    ax2.plot(sr.constants.PERIODS, cim_corr_res_std[sr.constants.PSA_KEYS], label="Std", c="green")
